# 🧠 NeuraQuiz AI — Complete GPU Pipeline
**Intelligent Reading Comprehension & Quiz Generation System**

This notebook runs the entire training pipeline on your **NVIDIA RTX 5070 Ti** GPU.

| Step | Description | Time |
|------|-------------|------|
| 1 | GPU & Environment Check | ~5 sec |
| 2 | Preprocessing (Feature Engineering on GPU) | ~6 min |
| 3 | EDA (Exploratory Data Analysis) | ~10 sec |
| 4 | Model A Training (9 models) | ~6 min |
| 5 | Model B Training (Distractor + SBERT) | ~2 min |
| 6 | Evaluation (Confusion Matrices + R²) | ~1 min |
| 7 | Launch Streamlit Dashboard | Interactive |

---
## Step 0: GPU & Environment Verification

In [ ]:
import torch
import sys

print("=" * 55)
print("  CUDA / GPU Diagnostic Report")
print("=" * 55)
print(f"  Python:        {sys.version.split()[0]}")
print(f"  PyTorch:       {torch.__version__}")
print(f"  CUDA available:{torch.cuda.is_available()}")
print(f"  CUDA version:  {torch.version.cuda}")
print(f"  cuDNN:         {torch.backends.cudnn.version()}")

if torch.cuda.is_available():
    dev = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(dev)
    free, total = torch.cuda.mem_get_info(dev)
    print(f"\n  GPU: {props.name}")
    print(f"  VRAM: {props.total_memory / 1e9:.2f} GB total | {free / 1e9:.2f} GB free")
    
    # Quick matmul test
    a = torch.randn(1000, 1000, device='cuda')
    c = a @ a
    torch.cuda.synchronize()
    print(f"  Matmul test:   PASS")
    print("\n  [OK] GPU is ready!")
else:
    print("\n  [FAIL] CUDA NOT AVAILABLE! Fix your GPU drivers.")
    raise SystemExit("Cannot proceed without GPU.")

---
## Step 1: Preprocessing (GPU-Accelerated Feature Engineering)
Generates TF-IDF features and GPU-computed cosine similarities for all 87,866 samples.

In [ ]:
%run src/preprocessing.py

---
## Step 2: Exploratory Data Analysis
Generates 7 publication-quality plots and summary statistics.

In [ ]:
%run src/eda.py

In [ ]:
# Display the EDA plots inline
import os
from IPython.display import Image, display

eda_dir = os.path.join(os.getcwd(), "results", "eda")
for img in sorted(os.listdir(eda_dir)):
    if img.endswith(".png"):
        print(f"\n--- {img.replace('_', ' ').replace('.png', '').title()} ---")
        display(Image(filename=os.path.join(eda_dir, img), width=800))

---
## Step 3: Train Model A — Answer Verification (9 Models on GPU)
Trains: XGBoost (CUDA), PyTorch MLP (CUDA), K-Means (CUDA), Logistic Regression, Random Forest, SVM, Naive Bayes, Label Propagation, and GPU Soft-Voting Ensemble.

In [ ]:
%run src/model_a_train.py

---
## Step 4: Train Model B — Distractor & Hint Generator (GPU)
Trains the XGBoost CUDA distractor ranker and loads Sentence-BERT on GPU for hint generation.

In [ ]:
%run src/model_b_train.py

---
## Step 5: Evaluation (Confusion Matrices + R² Score)
Evaluates all models on the held-out test set and saves confusion matrices for the dashboard.

In [ ]:
%run src/evaluate.py

---
## Step 6: View Final Results

In [ ]:
import pandas as pd

print("=" * 60)
print("  MODEL A — TEST SET RESULTS")
print("=" * 60)
df_a = pd.read_csv("results/model_a_test_results.csv")
display(df_a)

print("\n" + "=" * 60)
print("  MODEL B — TEST SET RESULTS")
print("=" * 60)
df_b = pd.read_csv("results/model_b_test_results.csv")
display(df_b)

print("\n" + "=" * 60)
print("  CONFUSION MATRICES")
print("=" * 60)
import json
with open("results/confusion_matrices.json") as f:
    cms = json.load(f)
for name, cm in cms.items():
    print(f"\n  {name}:")
    import numpy as np
    print(np.array(cm))

---
## Step 7: Launch the Streamlit Dashboard
Run this cell to launch the interactive quiz & analytics UI in your browser.

In [ ]:
!streamlit run ui/app.py

---
## Done!
All models are trained on the **RTX 5070 Ti** and the dashboard is ready.

**Models trained:** XGBoost (CUDA), PyTorch MLP (CUDA), K-Means (CUDA), Logistic Regression, Random Forest, SVM, Naive Bayes, Label Propagation, Soft-Voting Ensemble

**Artifacts saved to:**
- `models/model_a/` — All Model A classifiers
- `models/model_b/` — Distractor ranker + SBERT
- `results/` — CSVs, confusion matrices, EDA plots